# 📚 Importing Libraries

In [ ]:
import pandas as pd
import os
import matplotlib.pyplot as plt

# 🛠️ Auxiliary Functions

In [ ]:
def load_simulation_data(N, avg_k, C, drop_node=True):
    """
    📂 load_simulation_data(int N, int avg_k, int C, bool drop_node=True)

    Loads all CSV files generated by the simulation from the directory:
    ./data/N{N}/k{avg_k}/C{C}/

    Each file is read into a DataFrame with the following columns:
    ["p", "m1", "m2", "m3", "m4", "s", "node1", "node2", "t"]

    Args:
        N (int): network size
        avg_k (int): average degree
        C (int): maximum cost
        drop_node (bool, optional): if True, drops "node1" and "node2" columns. Default is True.

    Returns:
        list[pd.DataFrame]: list of DataFrames loaded from all CSV files
    """
    data_dir = f'./data/N{N}/k{avg_k}/C{C}/'
    dfs = []

    for file_name in os.listdir(data_dir):
        df = pd.read_csv(
            os.path.join(data_dir, file_name), 
            sep=' ', 
            names=["p", "m1", "m2", "m3", "m4", "s", "node1", "node2", "t"],
        )
        if drop_node is True:
            df.drop(['node1', 'node2'], axis=1, inplace=True)
        dfs.append(df)
    
    return dfs


def compute_average_curves(N, avg_k, C):
    """
    📊 compute_average_curves(int N, int avg_k, int C)

    Loads all simulation CSVs for a given configuration and computes
    the mean and standard deviation of all metrics grouped by "p".

    Args:
        N (int): network size
        avg_k (int): average degree
        C (int): maximum cost

    Returns:
        pd.DataFrame: DataFrame with mean and standard deviation 
                      for each value of "p".
                      Columns: ["p", "m1", "m2", "m3", "m4", "s", "t",
                                "m1_std", "m2_std", "m3_std", "m4_std", 
                                "s_std", "t_std"]
    """
    dfs = load_simulation_data(N, avg_k, C)
    combined_df = pd.concat(dfs)

    mean_df = combined_df.groupby('p').mean()
    std_df = combined_df.groupby('p').std()

    final_df = mean_df.merge(std_df, on='p', suffixes=('', '_std'))
    final_df = final_df.reset_index()
    
    return final_df


# 📥 Loading Data


In [ ]:
N = 2048
avg_k = 4
C = N

# Load simulation data
dfs = load_simulation_data(N=N, avg_k=avg_k, C=C)

# Compute averaged curves
final_df = compute_average_curves(N=N, avg_k=avg_k, C=C)